# Notebook 03: Evaluasi Baseline RAG — PubMedQA

Konfigurasi **Baseline RAG** tanpa teknik mitigasi apapun.
Retriever: **BM25 (keyword-based)**, tidak memerlukan embedding saat indexing.

## Konfigurasi
| Parameter | Nilai |
|-----------|-------|
| Query Rewriting | Tidak |
| Context Reranking | Tidak |
| Active Detection | Tidak |
| Retriever | BM25 (rank\_bm25) |
| LLM | llama3.2 via Ollama |
| Embedding | nomic-embed-text (hanya untuk RAGAS Answer Relevancy) |
| Dataset | PubMedQA pqa\_labeled |
| Sampel | 500 — ubah ke 1000 untuk evaluasi final |

## Alur
1. **DEMO** 5 sampel (verifikasi pipeline)
2. **Fase 1** Generate 500 jawaban + label accuracy (cepat)
3. **Fase 2** Evaluasi RAGAS per sampel (lambat, bisa semalam)
4. **Analisis** Ringkasan metrik + baris tabel skripsi

## 1. Impor Library

In [2]:
import os, sys, json, pickle, time, re, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Tuple
from pathlib import Path
from datetime import datetime
from collections import Counter

from rank_bm25 import BM25Okapi
import ollama
from datasets import load_dataset

# RAGAS dengan Ollama — gunakan old-style metrics (_Faithfulness, dll.)
# karena ragas.metrics.collections baru hanya support InstructorLLM (OpenAI)
warnings.filterwarnings('ignore', category=DeprecationWarning)
from ragas import EvaluationDataset, SingleTurnSample, evaluate, RunConfig
from ragas.metrics import (
    _Faithfulness,
    _ResponseRelevancy,
    _LLMContextPrecisionWithReference,
    _LLMContextRecall
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import OllamaLLM, OllamaEmbeddings

print('Semua library berhasil diimpor!')
print(f'Python: {sys.version.split()[0]} | NumPy: {np.__version__} | Pandas: {pd.__version__}')

C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Semua library berhasil diimpor!
Python: 3.11.9 | NumPy: 2.3.5 | Pandas: 2.3.3


## 2. Konfigurasi

In [3]:
LLM_MODEL   = 'llama3.2'
EMBED_MODEL = 'nomic-embed-text'  # Hanya untuk RAGAS, bukan retrieval

TOP_K_RETRIEVAL = 5

DATASET_NAME   = 'qiaojin/PubMedQA'
DATASET_SUBSET = 'pqa_labeled'

# CATATAN: Mulai 500. UBAH KE 1000 untuk evaluasi final thesis!
MAX_SAMPLES = 500

TEMPERATURE = 0.0
SEED        = 42

NOTEBOOK_DIR    = Path('.')
BM25_INDEX_PATH = NOTEBOOK_DIR / 'pubmedqa_bm25.pkl'
RESULTS_DIR     = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

CONFIG_NAME    = 'baseline'
PHASE1_PATH    = RESULTS_DIR / f'{CONFIG_NAME}_phase1_answers.json'
PHASE2_PATH    = RESULTS_DIR / f'{CONFIG_NAME}_phase2_ragas.json'
FINAL_CSV_PATH = RESULTS_DIR / f'{CONFIG_NAME}_results.csv'

print('Konfigurasi:')
print(f'  LLM        : {LLM_MODEL}')
print(f'  Embedding  : {EMBED_MODEL} (RAGAS only)')
print(f'  Retriever  : BM25')
print(f'  Top-K      : {TOP_K_RETRIEVAL}')
print(f'  Sampel     : {MAX_SAMPLES}')
print()
print('=' * 50)
print(f'  PENGINGAT: MAX_SAMPLES = {MAX_SAMPLES}')
print('  Ubah ke 1000 untuk evaluasi final thesis!')
print('=' * 50)

Konfigurasi:
  LLM        : llama3.2
  Embedding  : nomic-embed-text (RAGAS only)
  Retriever  : BM25
  Top-K      : 5
  Sampel     : 500

  PENGINGAT: MAX_SAMPLES = 500
  Ubah ke 1000 untuk evaluasi final thesis!


## 3. Data Classes dan Tokenizer BM25

In [4]:
@dataclass
class Document:
    text         : str
    pubid        : str
    question     : str
    section_label: str
    answer       : str
    decision     : str

@dataclass
class RetrievalResult:
    document: Document
    score   : float


def tokenize_bm25(text: str) -> List[str]:
    """Tokenizer untuk BM25: hapus tanda baca, lowercase, split spasi."""
    return re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower()).split()


# Test
sample_text = 'Does aspirin (75mg) reduce myocardial infarction risk?'
print(f'Tokenisasi BM25: {tokenize_bm25(sample_text)}')
print('Data classes dan tokenizer siap.')

Tokenisasi BM25: ['does', 'aspirin', '75mg', 'reduce', 'myocardial', 'infarction', 'risk']
Data classes dan tokenizer siap.


## 4. Muat Dataset dan Bangun BM25 Index

BM25 tidak memerlukan embedding untuk indexing — jauh lebih cepat dari FAISS.
Index dibangun dari tokenisasi teks dokumen saja.

In [5]:
def load_pubmedqa(subset=DATASET_SUBSET, max_samples=MAX_SAMPLES):
    print(f'Memuat PubMedQA ({subset})...')
    dataset = load_dataset(DATASET_NAME, subset, trust_remote_code=True)
    data    = dataset['train']
    if max_samples and len(data) > max_samples:
        data = data.select(range(max_samples))
    print(f'Dimuat {len(data)} sampel')
    return data


def prepare_documents(data) -> List[Document]:
    """Ubah dataset PubMedQA menjadi list Document (satu per section abstrak)."""
    docs = []
    for item in data:
        pubid = str(item['pubid'])
        for ctx, label in zip(item['context']['contexts'], item['context']['labels']):
            docs.append(Document(
                text=ctx.strip(), pubid=pubid,
                question=item['question'], section_label=label,
                answer=item['long_answer'], decision=item['final_decision']
            ))
    print(f'Total potongan dokumen: {len(docs)}')
    return docs


def load_or_build_bm25(data) -> Tuple[BM25Okapi, List[Document]]:
    """Muat BM25 index dari file jika ada, atau bangun dari scratch."""
    if BM25_INDEX_PATH.exists():
        print(f'Memuat BM25 index dari {BM25_INDEX_PATH}...')
        with open(BM25_INDEX_PATH, 'rb') as f:
            saved = pickle.load(f)
        print(f'Dimuat: {len(saved["documents"])} dokumen')
        return saved['bm25'], saved['documents']
    else:
        print('Membangun BM25 index...')
        documents = prepare_documents(data)
        tokenized = [tokenize_bm25(d.text) for d in documents]
        bm25      = BM25Okapi(tokenized)
        with open(BM25_INDEX_PATH, 'wb') as f:
            pickle.dump({'bm25': bm25, 'documents': documents}, f)
        print(f'Index disimpan ke {BM25_INDEX_PATH}')
        return bm25, documents


t0 = time.time()
pubmedqa_data         = load_pubmedqa()
bm25_index, documents = load_or_build_bm25(pubmedqa_data)
print(f'Selesai dalam {time.time()-t0:.1f} detik')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Memuat PubMedQA (pqa_labeled)...
Dimuat 500 sampel
Memuat BM25 index dari pubmedqa_bm25.pkl...
Dimuat: 1706 dokumen
Selesai dalam 7.7 detik


## 5. Fungsi Retrieval Baseline (BM25, Tanpa Query Rewriting)

In [6]:
def retrieve_baseline(query: str, k: int = TOP_K_RETRIEVAL) -> List[RetrievalResult]:
    """
    Retrieval baseline menggunakan BM25 (exact keyword matching).
    Tidak ada query rewriting. Tidak ada reranking.

    Catatan untuk notebook Context Reranking nanti:
      retrieve_baseline(query, k=20) untuk ambil lebih banyak kandidat,
      kemudian CrossEncoder rerank ke top-5.
    """
    tokens = tokenize_bm25(query)
    scores = bm25_index.get_scores(tokens)
    top_k  = np.argsort(scores)[::-1][:k]
    return [RetrievalResult(document=documents[i], score=float(scores[i])) for i in top_k]


# Test retrieval
test_q = 'Does aspirin reduce the risk of myocardial infarction?'
test_r = retrieve_baseline(test_q)
print(f'Query: {test_q}')
print(f'Top-{TOP_K_RETRIEVAL} dokumen (BM25):')
for i, r in enumerate(test_r, 1):
    print(f'  [{i}] Score={r.score:.4f} | {r.document.section_label} | {r.document.text[:90]}...')

Query: Does aspirin reduce the risk of myocardial infarction?
Top-5 dokumen (BM25):
  [1] Score=23.4945 | DESIGN | Within a prospective, population-based cohort study individuals without history of myocard...
  [2] Score=20.3167 | METHODS | By use of the Cooperative Cardiovascular Project database (a retrospective medical record ...
  [3] Score=19.3056 | METHODS | Of the 9681 women and 8888 men who attended risk assessment from 1967-1991, with follow-up...
  [4] Score=18.0483 | STUDY DESIGN | All patients between the ages of 80 to 89 years undergoing carotid endarterectomy during a...
  [5] Score=17.2286 | OBJECTIVE | To examine the effect of a weekend hospitalization on the timing and incidence of intensiv...


## 6. Prompt Generasi dan Fungsi Generate

In [7]:
GENERATION_PROMPT = (
    'You are a medical research assistant. '
    'Answer a biomedical yes/no/maybe question based solely on the provided scientific abstracts.\n\n'
    'Context from medical literature:\n{context}\n\n'
    'Question: {question}\n\n'
    'Instructions:\n'
    '- Carefully read the context and assess whether it supports or refutes the question.\n'
    '- Provide a brief explanation (2-3 sentences) using ONLY the information above.\n'
    '- End your response with EXACTLY ONE of these words on its own line: yes, no, or maybe.\n'
    '  - yes   : the evidence supports the hypothesis, even if not perfectly conclusive\n'
    '  - no    : the evidence refutes or does not support the hypothesis\n'
    '  - maybe : ONLY if the evidence is directly contradictory (some findings say yes,\n'
    '            others say no), or if the context contains no relevant information at all\n'
    '- IMPORTANT: If the evidence leans in one direction, even partially, choose yes or no.\n'
    '  Do NOT use maybe simply because the evidence is limited or not 100% certain.\n\n'
    'Answer:'
)


def generate_baseline_answer(query: str, retrieved: List[RetrievalResult]) -> str:
    """Generate jawaban. Baseline: tanpa query rewriting, tanpa reranking."""
    context = '\n\n'.join(
        f'[{i}] ({r.document.section_label}): {r.document.text}'
        for i, r in enumerate(retrieved, 1)
    )
    response = ollama.generate(
        model=LLM_MODEL,
        prompt=GENERATION_PROMPT.format(context=context, question=query),
        options={'temperature': TEMPERATURE, 'seed': SEED, 'num_predict': 300}
    )
    return response['response'].strip()


# Test
test_ans = generate_baseline_answer(test_q, test_r)
print('Output generation (prompt v2 - tighter maybe):')
print('-' * 60)
print(test_ans)
print('-' * 60)
print(f'Label: {extract_label(test_ans)!r}')

Output generation (prompt v2 - tighter maybe):
------------------------------------------------------------
Based on the provided scientific abstracts, there is no direct evidence to support the hypothesis that aspirin reduces the risk of myocardial infarction. The context primarily focuses on studies related to cardiovascular events, carotid endarterectomy, and hospitalization patterns, but none of these studies directly investigate the effect of aspirin on myocardial infarction risk.

yes
------------------------------------------------------------


NameError: name 'extract_label' is not defined

## 7. Ekstraksi Label yes/no/maybe

In [8]:
def extract_label(answer: str) -> str:
    """
    Ekstrak prediksi yes/no/maybe dari teks jawaban.
    Strategi (berurutan hingga ditemukan):
      1. Kata standalone di 3 baris terakhir (non-kosong)
      2. Kata standalone di seluruh teks
      3. Default ke 'maybe'
    """
    lines = [l.strip().lower() for l in answer.split('\n') if l.strip()]
    for line in reversed(lines[-3:]):
        word = re.sub(r'[^a-z]', '', line)
        if word in ('yes', 'no', 'maybe'):
            return word
    for label in ('yes', 'no', 'maybe'):
        if re.search(r'\b' + label + r'\b', answer.lower()):
            return label
    return 'maybe'


cases = [
    ('Strong evidence.\nyes',    'yes'),
    ('No effect found.\nno',     'no'),
    ('Mixed results.\nmaybe',    'maybe'),
    ('Verdict: yes.',             'yes'),
    ('Totally unclear.',          'maybe'),
]
print('Unit test extract_label:')
all_ok = True
for txt, exp in cases:
    pred = extract_label(txt)
    ok   = pred == exp
    all_ok = all_ok and ok
    print(f'  [{"PASS" if ok else "FAIL"}] pred={pred!r} expected={exp!r}')
print(f'\nSemua lulus: {all_ok}')
print(f'Label dari test answer: {extract_label(test_ans)!r}')

Unit test extract_label:
  [PASS] pred='yes' expected='yes'
  [PASS] pred='no' expected='no'
  [PASS] pred='maybe' expected='maybe'
  [PASS] pred='yes' expected='yes'
  [PASS] pred='maybe' expected='maybe'

Semua lulus: True
Label dari test answer: 'yes'


## 8. Setup RAGAS dengan Ollama (Tanpa OpenAI)

**Perbaikan `TimeoutError`**: RAGAS default `max_workers=16` — artinya 16 request dikirim ke Ollama bersamaan. Ollama hanya handle 1 sekaligus, sehingga request pertama antri dan timeout.

Solusi: `max_workers=1` (eksekusi sekuensial, tidak ada antrean).

In [9]:
ragas_llm = LangchainLLMWrapper(OllamaLLM(model=LLM_MODEL, temperature=0))
ragas_emb = LangchainEmbeddingsWrapper(OllamaEmbeddings(model=EMBED_MODEL))

# max_workers=1: evaluasi sampel sekuensial (mencegah antrean ke Ollama)
ragas_run_config = RunConfig(
    timeout     = 300,
    max_retries = 5,
    max_wait    = 120,
    max_workers = 1,
    log_tenacity= False,
)

faithfulness_m   = _Faithfulness(llm=ragas_llm)
ans_relevancy_m  = _ResponseRelevancy(llm=ragas_llm, embeddings=ragas_emb)
ctx_precision_m  = _LLMContextPrecisionWithReference(llm=ragas_llm)
ctx_recall_m     = _LLMContextRecall(llm=ragas_llm)

# Mapping: (key di hasil, metric object, nama kolom di pandas output)
METRIC_MAP = [
    ('faithfulness',       faithfulness_m,  'faithfulness'),
    ('answer_relevancy',   ans_relevancy_m, 'answer_relevancy'),
    ('context_precision',  ctx_precision_m, 'llm_context_precision_with_reference'),
    ('context_recall',     ctx_recall_m,    'context_recall'),
]

print('RAGAS siap (tanpa OpenAI, max_workers=1, per-metric try/except):')
for key, m, _ in METRIC_MAP:
    print(f'  - {key}')


def evaluate_ragas_single(question: str, answer: str,
                          contexts: List[str], reference: str) -> Dict:
    """
    Evaluasi RAGAS untuk 1 sampel, tiap metrik dievaluasi sendiri-sendiri.
    Jika satu metrik gagal (timeout / OutputParserException / dll),
    metrik lain tetap dievaluasi dan tidak ikut jadi NaN.
    """
    nan = float('nan')
    results = {key: nan for key, _, _ in METRIC_MAP}

    sample  = SingleTurnSample(
        user_input=question, response=answer,
        retrieved_contexts=contexts, reference=reference
    )
    dataset = EvaluationDataset(samples=[sample])

    for key, metric, col_name in METRIC_MAP:
        try:
            result = evaluate(
                dataset, metrics=[metric],
                llm=ragas_llm, embeddings=ragas_emb,
                run_config=ragas_run_config,
                raise_exceptions=False, show_progress=False
            )
            row   = result.to_pandas().iloc[0]
            val   = row.get(col_name, nan)
            results[key] = float(val) if val is not None else nan
        except Exception as e:
            # Tangkap TimeoutError, OutputParserException, dll per-metrik
            print(f'  [{key} SKIP] {type(e).__name__}: {str(e)[:80]}')

    return results

RAGAS siap (tanpa OpenAI, max_workers=1, per-metric try/except):
  - faithfulness
  - answer_relevancy
  - context_precision
  - context_recall


---
## Setup RAGAS Lightweight -- Faithfulness + Context Recall Only

Versi 2-metrik untuk menghindari NaN tinggi pada `context_precision` (89% NaN)
dan `answer_relevancy` (72% NaN) yang disebabkan oleh ketidakmampuan llama3.2 lokal
mengikuti format JSON strict yang dibutuhkan kedua metrik tersebut.

| Metrik | NaN rate (90 sampel) | Dipakai? |
|--------|----------------------|---------|
| Faithfulness | 63% | **Ya** |
| Answer Relevancy | 72% | Tidak |
| Context Precision | 89% | Tidak |
| Context Recall | 8% | **Ya** |

In [10]:
METRIC_MAP_FAST = [
    ('faithfulness',   faithfulness_m,  'faithfulness'),
    ('context_recall', ctx_recall_m,    'context_recall'),
]

print('RAGAS Lightweight (2 metrik):')
for key, m, _ in METRIC_MAP_FAST:
    print(f'  - {key}')


def evaluate_ragas_fast(question: str, answer: str,
                        contexts: List[str], reference: str) -> Dict:
    """Evaluasi RAGAS hanya Faithfulness + Context Recall.
    Tiap metrik dievaluasi sendiri agar satu gagal tidak drag lainnya.
    """
    nan = float('nan')
    results = {key: nan for key, _, _ in METRIC_MAP_FAST}

    sample  = SingleTurnSample(
        user_input=question, response=answer,
        retrieved_contexts=contexts, reference=reference
    )
    dataset = EvaluationDataset(samples=[sample])

    for key, metric, col_name in METRIC_MAP_FAST:
        try:
            result = evaluate(
                dataset, metrics=[metric],
                llm=ragas_llm, embeddings=ragas_emb,
                run_config=ragas_run_config,
                raise_exceptions=False, show_progress=False
            )
            row = result.to_pandas().iloc[0]
            val = row.get(col_name, nan)
            results[key] = float(val) if val is not None else nan
        except Exception as e:
            print(f'  [{key} SKIP] {type(e).__name__}: {str(e)[:80]}')

    return results

RAGAS Lightweight (2 metrik):
  - faithfulness
  - context_recall


---
## Setup Evaluator Custom (Zero-NaN)

RAGAS Faithfulness gagal karena LLM lokal tidak bisa menghasilkan JSON list of claims
secara konsisten. Solusi: **bypass RAGAS chain sepenuhnya** dengan evaluator custom.

### Perbedaan pendekatan
| | RAGAS | Custom (ini) |
|---|---|---|
| Ekstraksi klaim | LLM generate JSON list | `re.split` langsung (deterministik) |
| Verifikasi | LLM verify per claim | LLM yes/no per kalimat |
| NaN possible? | Ya (JSON parse fail) | **Tidak** |
| Prompt complexity | Tinggi (chain) | Rendah (1 pertanyaan) |

**Output: selalu float 0.0-1.0, tidak pernah NaN.**

In [5]:
def _split_sentences(text: str) -> List[str]:
    """Pecah teks menjadi kalimat. Filter kalimat terlalu pendek (<15 char)."""
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in parts if len(s.strip()) >= 15]


def _llm_yes_no(prompt: str) -> bool:
    """Tanya LLM ya/tidak. Return True=yes, False=no. Fallback False jika gagal."""
    try:
        resp = ollama.generate(
            model=LLM_MODEL, prompt=prompt,
            options={'temperature': 0, 'seed': SEED, 'num_predict': 10}
        )
        return 'yes' in resp['response'].strip().lower()[:15]
    except Exception:
        return False  # konservatif: kalau gagal anggap "no"


# --- Metrik 1: Faithfulness ---
def compute_faithfulness(answer: str, contexts: List[str]) -> float:
    """Faithfulness: fraksi kalimat jawaban yang didukung konteks. 0.0-1.0, no NaN."""
    sentences = _split_sentences(answer)
    if not sentences:
        return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = (
        'Context:\n{ctx}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement directly supported by the context above? '
        'Answer with only "yes" or "no".'
    )
    supported = sum(
        1 for s in sentences
        if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s))
    )
    return supported / len(sentences)


# --- Metrik 2: Context Recall ---
def compute_context_recall(reference: str, contexts: List[str]) -> float:
    """Context Recall: fraksi fakta reference yang tercakup konteks. 0.0-1.0, no NaN."""
    sentences = _split_sentences(reference)
    if not sentences:
        return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = (
        'Context:\n{ctx}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement supported by the context above? '
        'Answer with only "yes" or "no".'
    )
    covered = sum(
        1 for s in sentences
        if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s))
    )
    return covered / len(sentences)


# --- Metrik 3: Answer Relevancy (BARU) ---
def compute_answer_relevancy(question: str, answer: str) -> float:
    """
    Answer Relevancy: fraksi kalimat jawaban yang relevan dengan pertanyaan.
    Mengukur apakah jawaban benar-benar menjawab pertanyaan (bukan off-topic).
    0.0-1.0, no NaN.
    """
    sentences = _split_sentences(answer)
    if not sentences:
        return 0.0
    prompt_tmpl = (
        'Question: {question}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement relevant to answering the question above? '
        'Answer with only "yes" or "no".'
    )
    relevant = sum(
        1 for s in sentences
        if _llm_yes_no(prompt_tmpl.format(question=question, sent=s))
    )
    return relevant / len(sentences)


# --- Metrik 4: Context Precision (BARU) ---
def compute_context_precision(question: str, contexts: List[str], reference: str) -> float:
    """
    Context Precision (Average Precision): apakah konteks relevan di rank atas?
    Formula: AP = sum(Precision@k * rel_k) / total_relevant
    0.0-1.0, no NaN.
    """
    if not contexts:
        return 0.0
    prompt_tmpl = (
        'Question: {question}\n\n'
        'Ground truth answer: {reference}\n\n'
        'Retrieved context: {ctx}\n\n'
        'Does this context contain information useful for correctly answering '
        'the question based on the ground truth? Answer with only "yes" or "no".'
    )
    relevance = []
    for ctx in contexts:
        is_rel = _llm_yes_no(prompt_tmpl.format(
            question=question, reference=reference[:300], ctx=ctx[:400]
        ))
        relevance.append(1 if is_rel else 0)

    total_relevant = sum(relevance)
    if total_relevant == 0:
        return 0.0

    precision_sum = 0.0
    relevant_count = 0
    for k, rel in enumerate(relevance):
        if rel:
            relevant_count += 1
            precision_at_k = relevant_count / (k + 1)
            precision_sum += precision_at_k
    return precision_sum / total_relevant


# --- Wrapper ---
def evaluate_custom(question: str, answer: str,
                    contexts: List[str], reference: str) -> Dict:
    """Wrapper evaluasi 1 sampel dengan 4 metrik. Selalu return dict tanpa NaN."""
    return {
        'faithfulness'      : compute_faithfulness(answer, contexts),
        'context_recall'    : compute_context_recall(reference, contexts),
        'answer_relevancy'  : compute_answer_relevancy(question, answer),
        'context_precision' : compute_context_precision(question, contexts, reference),
    }


# Smoke test
_test_ctx  = ['Aspirin reduces blood clotting and is used for heart attack prevention.']
_test_ans  = 'Aspirin helps prevent heart attacks. It works by reducing clotting.'
_test_ref  = 'Aspirin is used for heart attack prevention by reducing blood clotting.'
_r = evaluate_custom('Does aspirin prevent heart attacks?', _test_ans, _test_ctx, _test_ref)
print('Smoke test evaluate_custom (4 metrik):')
print(f'  faithfulness      = {_r["faithfulness"]:.3f}')
print(f'  context_recall    = {_r["context_recall"]:.3f}')
print(f'  answer_relevancy  = {_r["answer_relevancy"]:.3f}')
print(f'  context_precision = {_r["context_precision"]:.3f}')
print('Zero-NaN evaluator siap (4 metrik).')


Smoke test evaluate_custom (4 metrik):
  faithfulness      = 1.000
  context_recall    = 1.000
  answer_relevancy  = 1.000
  context_precision = 1.000
Zero-NaN evaluator siap (4 metrik).


---
## DEMO: Uji Coba 5 Sampel

Jalankan ini dulu untuk memverifikasi seluruh pipeline berjalan dengan benar.

> Estimasi: ~3-10 menit (tergantung kecepatan Ollama)

In [9]:
DEMO_SIZE    = 5
demo_results = []

print(f'DEMO: {DEMO_SIZE} sampel pertama')
print('=' * 65)

for i in range(DEMO_SIZE):
    s         = pubmedqa_data[i]
    q         = s['question']
    gt        = s['final_decision']
    ref       = s['long_answer']

    print(f'\n[{i+1}/{DEMO_SIZE}] PubID: {s["pubid"]}')
    print(f'  Q  : {q[:80]}...')
    print(f'  GT : {gt}')

    t0        = time.time()
    retrieved = retrieve_baseline(q)
    t_ret     = time.time() - t0

    t0     = time.time()
    answer = generate_baseline_answer(q, retrieved)
    t_gen  = time.time() - t0

    predicted = extract_label(answer)
    correct   = predicted == gt

    t0      = time.time()
    ctxs    = [r.document.text for r in retrieved]
    scores  = evaluate_ragas_single(q, answer, ctxs, ref)
    t_ragas = time.time() - t0

    demo_results.append({
        'idx': i, 'pubid': str(s['pubid']), 'question': q,
        'ground_truth': gt, 'predicted_label': predicted,
        'is_correct': correct, 'answer': answer,
        'contexts': ctxs, 'reference': ref, **scores,
        't_retrieval': round(t_ret,2), 't_generation': round(t_gen,2), 't_ragas': round(t_ragas,2)
    })

    verdict = 'BENAR' if correct else 'SALAH'
    print(f'  Pred : {predicted} [{verdict}]')
    print(f'  Ans  : {answer[:100]}...')
    print(f'  RAGAS: faith={scores["faithfulness"]:.3f} | '
          f'rel={scores["answer_relevancy"]:.3f} | '
          f'cp={scores["context_precision"]:.3f} | '
          f'cr={scores["context_recall"]:.3f}')
    print(f'  Waktu: ret={t_ret:.1f}s | gen={t_gen:.1f}s | ragas={t_ragas:.1f}s')

n_ok = sum(r['is_correct'] for r in demo_results)
print(f'\n{"="*65}')
print(f'RINGKASAN DEMO ({DEMO_SIZE} sampel):')
print(f'  Label Accuracy    : {n_ok}/{DEMO_SIZE} = {n_ok/DEMO_SIZE:.1%}')
print(f'  Hallucination Rate: {(DEMO_SIZE-n_ok)/DEMO_SIZE:.1%}')
for col in ['faithfulness','answer_relevancy','context_precision','context_recall']:
    print(f'  Avg {col:<22}: {np.nanmean([r[col] for r in demo_results]):.3f}')
print(f'{"="*65}')

DEMO: 5 sampel pertama

[1/5] PubID: 21645374
  Q  : Do mitochondria play a role in remodelling lace plant leaves during programmed c...
  GT : yes
  Pred : yes [BENAR]
  Ans  : Based on the provided scientific abstracts, it appears that mitochondria play a role in programmed c...
  RAGAS: faith=1.000 | rel=0.964 | cp=0.950 | cr=0.750
  Waktu: ret=0.0s | gen=60.9s | ragas=638.2s

[2/5] PubID: 16418930
  Q  : Landolt C and snellen e acuity: differences in strabismus amblyopia?...
  GT : no
  Pred : yes [SALAH]
  Ans  : The study abstracts suggest that there are small differences between Landolt C acuity (LR) and Snell...
  RAGAS: faith=0.889 | rel=0.486 | cp=1.000 | cr=0.800
  Waktu: ret=0.0s | gen=56.3s | ragas=625.8s

[3/5] PubID: 9488747
  Q  : Syncope during bathing in infants, a pediatric form of water-induced urticaria?...
  GT : yes


Exception raised in Job[1]: OutputParserException(Invalid json output: Does syncope during bathing in infants support a pediatric form of water-induced urticaria?
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )
Exception raised in Job[2]: OutputParserException(Invalid json output: The provided context was not useful in arriving at the given answer. The context discusses a different medical condition (apparent life-threatening events in infants) and does not provide any relevant information about syncope during bathing or aquagenic maladies.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )


  Pred : yes [BENAR]
  Ans  : The question of syncope during bathing in infants, a pediatric form of water-induced urticaria, can ...
  RAGAS: faith=1.000 | rel=nan | cp=nan | cr=0.286
  Waktu: ret=0.0s | gen=58.6s | ragas=763.4s

[4/5] PubID: 17208539
  Q  : Are the long-term results of the transanal pull-through equal to those of the tr...
  GT : no
  Pred : maybe [SALAH]
  Ans  : The study comparing TERPT and ABD pull-through for Hirschsprung disease does not provide direct info...
  RAGAS: faith=0.333 | rel=0.000 | cp=0.887 | cr=0.750
  Waktu: ret=0.0s | gen=74.1s | ragas=642.2s

[5/5] PubID: 10808977
  Q  : Can tailored interventions increase mammography use among HMO women?...
  GT : yes


Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\asyncio\events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000020FFF5C5CC0> is already entered
Task was destroyed but it is pending!
task: <Task pending name='Task-8050' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-8051' coro=<Kernel.shell_main() running at C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel\kernelbase.py:590> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Users\R

  Pred : yes [BENAR]
  Ans  : Based on the provided scientific abstracts, tailored interventions can increase mammography use amon...
  RAGAS: faith=nan | rel=nan | cp=nan | cr=0.000
  Waktu: ret=0.0s | gen=40.9s | ragas=832.6s

RINGKASAN DEMO (5 sampel):
  Label Accuracy    : 3/5 = 60.0%
  Hallucination Rate: 40.0%
  Avg faithfulness          : 0.806
  Avg answer_relevancy      : 0.483
  Avg context_precision     : 0.946
  Avg context_recall        : 0.517


---
## Fase 1: Generate Semua Jawaban (500 Sampel)

Hanya retrieval + generation, tanpa RAGAS. Disimpan inkremental setiap 10 sampel.

> Estimasi: ~25-50 menit untuk 500 sampel

In [10]:
if PHASE1_PATH.exists():
    with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
        phase1_results = json.load(f)['results']
    start_from = len(phase1_results)
    print(f'Resume Fase 1: {start_from}/{MAX_SAMPLES} sudah selesai.')
else:
    phase1_results, start_from = [], 0
    print(f'Memulai Fase 1: {MAX_SAMPLES} sampel.')

if start_from < MAX_SAMPLES:
    print(f'Memproses {MAX_SAMPLES - start_from} sampel tersisa...\n')
    t_start = time.time()
    for i in range(start_from, MAX_SAMPLES):
        s          = pubmedqa_data[i]
        q, gt, ref = s['question'], s['final_decision'], s['long_answer']
        retrieved  = retrieve_baseline(q)
        answer     = generate_baseline_answer(q, retrieved)
        predicted  = extract_label(answer)
        phase1_results.append({
            'idx': i, 'pubid': str(s['pubid']), 'question': q,
            'ground_truth': gt, 'predicted_label': predicted,
            'is_correct': predicted == gt, 'answer': answer,
            'contexts': [r.document.text for r in retrieved],
            'reference': ref, 'retrieval_scores': [r.score for r in retrieved],
        })
        if (i + 1) % 10 == 0 or i == MAX_SAMPLES - 1:
            with open(PHASE1_PATH, 'w', encoding='utf-8') as f:
                json.dump({'config': CONFIG_NAME,
                           'timestamp': datetime.now().isoformat(),
                           'max_samples': MAX_SAMPLES, 'completed': i+1,
                           'results': phase1_results}, f, indent=2, ensure_ascii=False)
            done = i + 1
            acc  = sum(r['is_correct'] for r in phase1_results) / done
            eta  = (time.time()-t_start) / done * (MAX_SAMPLES-done) / 60
            print(f'  [{done:3d}/{MAX_SAMPLES}] Akurasi: {acc:.1%} | pred={predicted}, gt={gt} | ETA {eta:.1f} mnt')
    print(f'\nFase 1 selesai! Disimpan ke {PHASE1_PATH}')
else:
    print(f'Fase 1 sudah selesai ({MAX_SAMPLES} sampel).')

Memulai Fase 1: 500 sampel.
Memproses 500 sampel tersisa...

  [ 10/500] Akurasi: 50.0% | pred=yes, gt=yes | ETA 433.6 mnt
  [ 20/500] Akurasi: 60.0% | pred=yes, gt=yes | ETA 391.4 mnt
  [ 30/500] Akurasi: 63.3% | pred=maybe, gt=yes | ETA 365.9 mnt
  [ 40/500] Akurasi: 55.0% | pred=yes, gt=no | ETA 346.8 mnt
  [ 50/500] Akurasi: 62.0% | pred=no, gt=no | ETA 335.3 mnt
  [ 60/500] Akurasi: 58.3% | pred=yes, gt=yes | ETA 323.5 mnt
  [ 70/500] Akurasi: 58.6% | pred=yes, gt=yes | ETA 318.8 mnt
  [ 80/500] Akurasi: 58.8% | pred=yes, gt=yes | ETA 311.0 mnt
  [ 90/500] Akurasi: 60.0% | pred=yes, gt=maybe | ETA 303.5 mnt
  [100/500] Akurasi: 61.0% | pred=yes, gt=yes | ETA 299.9 mnt
  [110/500] Akurasi: 61.8% | pred=yes, gt=yes | ETA 293.6 mnt
  [120/500] Akurasi: 60.0% | pred=yes, gt=yes | ETA 285.7 mnt
  [130/500] Akurasi: 58.5% | pred=yes, gt=maybe | ETA 278.3 mnt
  [140/500] Akurasi: 59.3% | pred=yes, gt=yes | ETA 270.3 mnt
  [150/500] Akurasi: 59.3% | pred=no, gt=no | ETA 262.2 mnt
  [160/5

### Analisis Fase 1 (Label Accuracy)

In [6]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']
n         = len(results_p1)
n_correct = sum(r['is_correct'] for r in results_p1)
gts       = [r['ground_truth']    for r in results_p1]
preds     = [r['predicted_label'] for r in results_p1]

print(f'ANALISIS FASE 1 — {n} sampel')
print('=' * 50)
print(f'Label Accuracy    : {n_correct}/{n} = {n_correct/n:.1%}')
print(f'Hallucination Rate: {(n-n_correct)/n:.1%}\n')
print(f'  {"Label":<8} | {"Ground Truth":>12} | {"Prediksi":>12}')
print(f'  {"-"*8}-+{"-"*14}-+{"-"*12}')
for lbl in ['yes','no','maybe']:
    g, p = gts.count(lbl), preds.count(lbl)
    print(f'  {lbl:<8} | {g:>10} ({g/n:.0%}) | {p:>10} ({p/n:.0%})')
print('\nConfusion Matrix (baris=GT, kolom=Prediksi):')
lbls = ['yes','no','maybe']
print('  ' + f'{"GT/Pred":>8}' + ''.join(f'{l:>8}' for l in lbls))
for gt_l in lbls:
    row = f'  {gt_l:>8}'
    for pr_l in lbls:
        cnt = sum(1 for r in results_p1 if r['ground_truth']==gt_l and r['predicted_label']==pr_l)
        row += f'{cnt:>8}'
    print(row)

ANALISIS FASE 1 — 500 sampel
Label Accuracy    : 278/500 = 55.6%
Hallucination Rate: 44.4%

  Label    | Ground Truth |     Prediksi
  ---------+---------------+------------
  yes      |        275 (55%) |        431 (86%)
  no       |        159 (32%) |         50 (10%)
  maybe    |         66 (13%) |         19 (4%)

Confusion Matrix (baris=GT, kolom=Prediksi):
   GT/Pred     yes      no   maybe
       yes     249      16      10
        no     123      28       8
     maybe      59       6       1


---
## Fase 2: Evaluasi RAGAS

Jalankan setelah Fase 1 selesai. Mendukung resume — aman dihentikan kapan saja.

> Estimasi: ~2-5 menit per sampel. Disarankan dijalankan semalam.

In [ ]:
if PHASE2_PATH.exists():
    with open(PHASE2_PATH, 'r', encoding='utf-8') as f:
        phase2_results = json.load(f)['results']
    done_idx = {r['idx'] for r in phase2_results}
    print(f'Resume Fase 2: {len(done_idx)}/{len(results_p1)} sudah dievaluasi.')
else:
    phase2_results, done_idx = [], set()
    print(f'Memulai Fase 2: {len(results_p1)} sampel.')

remaining = [r for r in results_p1 if r['idx'] not in done_idx]
print(f'Sisa: {len(remaining)} sampel | Estimasi ~{len(remaining)*3/60:.1f} jam\n')

t_p2 = time.time()
for i, r in enumerate(remaining):
    scores = evaluate_ragas_single(r['question'], r['answer'], r['contexts'], r['reference'])
    phase2_results.append({
        'idx': r['idx'], 'ground_truth': r['ground_truth'],
        'predicted_label': r['predicted_label'], 'is_correct': r['is_correct'],
        **scores
    })
    if (i + 1) % 5 == 0 or i == len(remaining) - 1:
        with open(PHASE2_PATH, 'w', encoding='utf-8') as f:
            json.dump({'config': CONFIG_NAME, 'timestamp': datetime.now().isoformat(),
                       'results': phase2_results}, f, indent=2, ensure_ascii=False)
        done  = i + 1
        total = len(remaining)
        eta   = (time.time()-t_p2)/done*(total-done)/60 if done < total else 0
        avg_f = np.nanmean([x['faithfulness'] for x in phase2_results])
        print(f'  [{done:3d}/{total}] idx={r["idx"]} | '
              f'faith={scores["faithfulness"]:.3f} | avg_f={avg_f:.3f} | ETA {eta:.1f} mnt')
print(f'\nFase 2 selesai! Disimpan ke {PHASE2_PATH}')

---
## Fase 2 Fast -- 100 Sampel (Faithfulness + Context Recall)

Gunakan `evaluate_ragas_fast` -- hanya 2 metrik reliable.
Resume-able, simpan ke `{CONFIG_NAME}_phase2_fast.json`.

> Estimasi: ~1.5-2.5 jam untuk 100 sampel

In [ ]:
MAX_RAGAS_SAMPLES = 100
PHASE2_FAST_PATH  = RESULTS_DIR / f'{CONFIG_NAME}_phase2_fast.json'

# Load Fase 1 (ambil MAX_RAGAS_SAMPLES pertama)
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    p1_all = json.load(f)['results']
p1_subset = p1_all[:MAX_RAGAS_SAMPLES]

if PHASE2_FAST_PATH.exists():
    with open(PHASE2_FAST_PATH, 'r', encoding='utf-8') as f:
        phase2_fast = json.load(f)['results']
    done_idx_fast = {r['idx'] for r in phase2_fast}
    print(f'Resume Fase 2 Fast: {len(done_idx_fast)}/{MAX_RAGAS_SAMPLES} sudah dievaluasi.')
else:
    phase2_fast, done_idx_fast = [], set()
    print(f'Memulai Fase 2 Fast: {MAX_RAGAS_SAMPLES} sampel (faithfulness + context_recall).')

remaining_fast = [r for r in p1_subset if r['idx'] not in done_idx_fast]
print(f'Sisa: {len(remaining_fast)} sampel | Estimasi ~{len(remaining_fast)*1.5/60:.1f} jam\n')

t_p2 = time.time()
for i, r in enumerate(remaining_fast):
    scores = evaluate_ragas_fast(r['question'], r['answer'], r['contexts'], r['reference'])
    phase2_fast.append({
        'idx': r['idx'], 'ground_truth': r['ground_truth'],
        'predicted_label': r['predicted_label'], 'is_correct': r['is_correct'],
        **scores
    })
    if (i + 1) % 5 == 0 or i == len(remaining_fast) - 1:
        with open(PHASE2_FAST_PATH, 'w', encoding='utf-8') as f:
            json.dump({'config': CONFIG_NAME, 'timestamp': datetime.now().isoformat(),
                       'max_samples': MAX_RAGAS_SAMPLES,
                       'metrics': ['faithfulness', 'context_recall'],
                       'results': phase2_fast}, f, indent=2, ensure_ascii=False)
        done  = i + 1
        total = len(remaining_fast)
        eta   = (time.time()-t_p2)/done*(total-done)/60 if done < total else 0
        avg_f = np.nanmean([x['faithfulness']   for x in phase2_fast])
        avg_r = np.nanmean([x['context_recall'] for x in phase2_fast])
        print(f'  [{done:3d}/{total}] idx={r["idx"]} | '
              f'faith={scores["faithfulness"]:.3f} | cr={scores["context_recall"]:.3f} | '
              f'avg_f={avg_f:.3f} | avg_cr={avg_r:.3f} | ETA {eta:.1f} mnt')

print(f'\nFase 2 Fast selesai! Disimpan ke {PHASE2_FAST_PATH}')

# Ringkasan
n_eval = len(phase2_fast)
avg_f  = np.nanmean([x['faithfulness']   for x in phase2_fast])
avg_cr = np.nanmean([x['context_recall'] for x in phase2_fast])
n_f_valid  = sum(1 for x in phase2_fast if x['faithfulness']  == x['faithfulness'])
n_cr_valid = sum(1 for x in phase2_fast if x['context_recall'] == x['context_recall'])
acc = sum(r['is_correct'] for r in phase2_fast) / n_eval
print(f'\nRingkasan ({n_eval} sampel):')
print(f'  Label Accuracy : {acc:.1%}')
print(f'  Faithfulness   : {avg_f:.4f}  (valid={n_f_valid}/{n_eval}, NaN={n_eval-n_f_valid})')
print(f'  Context Recall : {avg_cr:.4f} (valid={n_cr_valid}/{n_eval}, NaN={n_eval-n_cr_valid})')
print(f'\nBaris tabel skripsi:')
print(f'  | {CONFIG_NAME} | {acc:.3f} | {avg_f:.3f} | {avg_cr:.3f} |')

Memulai Fase 2 Fast: 100 sampel (faithfulness + context_recall).
Sisa: 100 sampel | Estimasi ~2.5 jam



Exception raised in Job[0]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
C:\Users\Ricky Wijaya\AppData\Local\Temp\ipykernel_19864\2708114915.py:38: RuntimeWarning: Mean of empty slice
  avg_f = np.nanmean([x['faithfulness']   for x in phase2_fast])


  [  5/100] idx=4 | faith=nan | cr=0.000 | avg_f=nan | avg_cr=0.575 | ETA 859.1 mnt


Exception raised in Job[0]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
C:\Users\Ricky Wijaya\AppData\Local\Temp\ipykernel_19864\2708114915.py:38: RuntimeWarning: Mean of empty slice
  avg_f = np.nanmean([x['faithfulness']   for x in phase2_fast])


  [ 10/100] idx=9 | faith=nan | cr=0.333 | avg_f=nan | avg_cr=0.515 | ETA 820.2 mnt


Exception raised in Job[0]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
C:\Users\Ricky Wijaya\AppData\Local\Temp\ipykernel_19864\2708114915.py:38: RuntimeWarning: Mean of empty slice
  avg_f = np.nanmean([x['faithfulness']   for x in phase2_fast])


  [ 15/100] idx=14 | faith=nan | cr=nan | avg_f=nan | avg_cr=0.553 | ETA 796.0 mnt


---
## Fase 2 Custom (Zero-NaN) -- 500 Sampel

Gunakan `evaluate_custom` -- dijamin tidak NaN. Resume-able.
Simpan ke `{CONFIG_NAME}_phase2_custom.json`.

> Estimasi: ~1-2 jam untuk 100 sampel (tiap sampel ~3-6 LLM calls singkat)

In [7]:
MAX_CUSTOM_SAMPLES  = 500
PHASE2_CUSTOM_PATH  = RESULTS_DIR / f'{CONFIG_NAME}_phase2_custom.json'

# Metrik yang harus ada di setiap sampel (untuk smart resume)
REQUIRED_METRICS = ['faithfulness', 'context_recall', 'answer_relevancy', 'context_precision']

with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    p1_custom = json.load(f)['results'][:MAX_CUSTOM_SAMPLES]

if PHASE2_CUSTOM_PATH.exists():
    with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
        p2_custom = json.load(f)['results']
    done_custom = {r['idx'] for r in p2_custom if all(m in r for m in REQUIRED_METRICS)}
    needs_upgrade = [r for r in p2_custom if not all(m in r for m in REQUIRED_METRICS)]
    print(f'Resume: {len(done_custom)}/{MAX_CUSTOM_SAMPLES} selesai dengan 4 metrik.')
    if needs_upgrade:
        print(f'Perlu upgrade: {len(needs_upgrade)} sampel (punya 2 metrik, butuh 2 metrik baru).')
else:
    p2_custom, done_custom, needs_upgrade = [], set(), []
    print(f'Mulai: {MAX_CUSTOM_SAMPLES} sampel (custom zero-NaN, 4 metrik).')

# --- TAHAP 1: Upgrade sampel yang sudah punya 2 metrik (faithfulness + context_recall) ---
if needs_upgrade:
    print(f'\nTahap 1: Upgrade {len(needs_upgrade)} sampel ke 4 metrik (hanya hitung 2 metrik baru)...')
    t_up = time.time()
    p1_lookup = {r['idx']: r for r in p1_custom}

    for i, r in enumerate(needs_upgrade):
        src = p1_lookup[r['idx']]
        if 'answer_relevancy' not in r:
            r['answer_relevancy'] = compute_answer_relevancy(src['question'], src['answer'])
        if 'context_precision' not in r:
            r['context_precision'] = compute_context_precision(
                src['question'], src['contexts'], src['reference']
            )
        done_custom.add(r['idx'])

        if (i + 1) % 5 == 0 or i == len(needs_upgrade) - 1:
            with open(PHASE2_CUSTOM_PATH, 'w', encoding='utf-8') as f:
                json.dump({
                    'config': CONFIG_NAME, 'timestamp': datetime.now().isoformat(),
                    'max_samples': MAX_CUSTOM_SAMPLES,
                    'metrics': REQUIRED_METRICS,
                    'evaluator': 'custom_zero_nan_4metrics',
                    'results': p2_custom
                }, f, indent=2, ensure_ascii=False)
            done  = i + 1
            eta   = (time.time()-t_up)/done*(len(needs_upgrade)-done)/60 if done < len(needs_upgrade) else 0
            print(f'  upgrade [{done:3d}/{len(needs_upgrade)}] idx={r["idx"]} | '
                  f'ar={r["answer_relevancy"]:.2f} cp={r["context_precision"]:.2f} | ETA {eta:.1f} mnt')
    print(f'Tahap 1 selesai.\n')

# --- TAHAP 2: Evaluasi sampel baru (4 metrik sekaligus) ---
remaining = [r for r in p1_custom if r['idx'] not in done_custom]
print(f'Tahap 2: Evaluasi {len(remaining)} sampel baru (4 metrik) | Estimasi ~{len(remaining)*1.0/60:.1f} jam\n')

t0 = time.time()
for i, r in enumerate(remaining):
    scores = evaluate_custom(r['question'], r['answer'], r['contexts'], r['reference'])
    p2_custom.append({
        'idx': r['idx'], 'ground_truth': r['ground_truth'],
        'predicted_label': r['predicted_label'], 'is_correct': r['is_correct'],
        **scores
    })
    if (i + 1) % 5 == 0 or i == len(remaining) - 1:
        with open(PHASE2_CUSTOM_PATH, 'w', encoding='utf-8') as f:
            json.dump({
                'config': CONFIG_NAME, 'timestamp': datetime.now().isoformat(),
                'max_samples': MAX_CUSTOM_SAMPLES,
                'metrics': REQUIRED_METRICS,
                'evaluator': 'custom_zero_nan_4metrics',
                'results': p2_custom
            }, f, indent=2, ensure_ascii=False)
        done  = i + 1
        total = len(remaining)
        eta   = (time.time()-t0)/done*(total-done)/60 if done < total else 0
        avg_f  = sum(x['faithfulness']      for x in p2_custom) / len(p2_custom)
        avg_cr = sum(x['context_recall']    for x in p2_custom) / len(p2_custom)
        avg_ar = sum(x['answer_relevancy']  for x in p2_custom) / len(p2_custom)
        avg_cp = sum(x['context_precision'] for x in p2_custom) / len(p2_custom)
        print(f'  [{done:3d}/{total}] idx={r["idx"]} | '
              f'f={scores["faithfulness"]:.2f} cr={scores["context_recall"]:.2f} '
              f'ar={scores["answer_relevancy"]:.2f} cp={scores["context_precision"]:.2f} | '
              f'avg: f={avg_f:.3f} cr={avg_cr:.3f} ar={avg_ar:.3f} cp={avg_cp:.3f} | ETA {eta:.1f} mnt')

print(f'\nSelesai! -> {PHASE2_CUSTOM_PATH}')

# --- Ringkasan akhir ---
n   = len(p2_custom)
acc = sum(r['is_correct'] for r in p2_custom) / n
avg_f  = sum(r['faithfulness']      for r in p2_custom) / n
avg_cr = sum(r['context_recall']    for r in p2_custom) / n
avg_ar = sum(r['answer_relevancy']  for r in p2_custom) / n
avg_cp = sum(r['context_precision'] for r in p2_custom) / n
print(f'\nRingkasan ({n} sampel, zero-NaN evaluator, 4 metrik):')
print(f'  Label Accuracy    : {acc:.1%}')
print(f'  Faithfulness      : {avg_f:.4f}')
print(f'  Context Recall    : {avg_cr:.4f}')
print(f'  Answer Relevancy  : {avg_ar:.4f}')
print(f'  Context Precision : {avg_cp:.4f}')
print(f'\nBaris tabel skripsi:')
print(f'  | {CONFIG_NAME.upper()} | {acc:.3f} | {avg_f:.3f} | {avg_cr:.3f} | {avg_ar:.3f} | {avg_cp:.3f} |')


Resume: 50/500 selesai dengan 4 metrik.
Perlu upgrade: 450 sampel (punya 2 metrik, butuh 2 metrik baru).

Tahap 1: Upgrade 450 sampel ke 4 metrik (hanya hitung 2 metrik baru)...
  upgrade [  5/450] idx=54 | ar=1.00 cp=1.00 | ETA 504.1 mnt
  upgrade [ 10/450] idx=59 | ar=1.00 cp=1.00 | ETA 515.6 mnt
  upgrade [ 15/450] idx=64 | ar=1.00 cp=0.81 | ETA 532.5 mnt
  upgrade [ 20/450] idx=69 | ar=1.00 cp=1.00 | ETA 552.8 mnt
  upgrade [ 25/450] idx=74 | ar=1.00 cp=1.00 | ETA 502.8 mnt
  upgrade [ 30/450] idx=79 | ar=1.00 cp=1.00 | ETA 462.7 mnt
  upgrade [ 35/450] idx=84 | ar=1.00 cp=1.00 | ETA 432.8 mnt
  upgrade [ 40/450] idx=89 | ar=1.00 cp=1.00 | ETA 408.0 mnt
  upgrade [ 45/450] idx=94 | ar=1.00 cp=0.81 | ETA 393.9 mnt
  upgrade [ 50/450] idx=99 | ar=1.00 cp=1.00 | ETA 378.5 mnt
  upgrade [ 55/450] idx=104 | ar=0.67 cp=0.70 | ETA 365.6 mnt
  upgrade [ 60/450] idx=109 | ar=1.00 cp=1.00 | ETA 353.6 mnt
  upgrade [ 65/450] idx=114 | ar=1.00 cp=1.00 | ETA 343.5 mnt
  upgrade [ 70/450] idx=11

---
## Hasil Akhir: Gabungkan Fase 1 + Fase 2 → CSV

In [ ]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']
with open(PHASE2_PATH, 'r', encoding='utf-8') as f:
    results_p2 = json.load(f)['results']

p2_lookup = {r['idx']: r for r in results_p2}

df = pd.DataFrame([{
    'idx': r['idx'], 'pubid': r['pubid'], 'question': r['question'],
    'ground_truth': r['ground_truth'], 'predicted_label': r['predicted_label'],
    'is_correct': r['is_correct'],
    'faithfulness'      : p2_lookup.get(r['idx'],{}).get('faithfulness',       float('nan')),
    'answer_relevancy'  : p2_lookup.get(r['idx'],{}).get('answer_relevancy',   float('nan')),
    'context_precision' : p2_lookup.get(r['idx'],{}).get('context_precision',  float('nan')),
    'context_recall'    : p2_lookup.get(r['idx'],{}).get('context_recall',     float('nan')),
} for r in results_p1])

df.to_csv(FINAL_CSV_PATH, index=False, encoding='utf-8')
print(f'Disimpan ke: {FINAL_CSV_PATH}')
print(f'Total: {len(df)} | RAGAS lengkap: {df["faithfulness"].notna().sum()}')
df.head(10)

## Ringkasan Metrik Evaluasi Baseline

In [ ]:
n, n_correct = len(df), int(df['is_correct'].sum())
label_acc  = n_correct / n
hallu_rate = 1 - label_acc
avg_f  = df['faithfulness'].mean()
avg_r  = df['answer_relevancy'].mean()
avg_cp = df['context_precision'].mean()
avg_cr = df['context_recall'].mean()

print('=' * 60)
print(f'  BASELINE (BM25) — {n} sampel')
print('=' * 60)
print(f'  {"Metrik":<32} {"Nilai":>10}')
print(f'  {"-"*42}')
print(f'  {"Label Accuracy":<32} {label_acc:>9.1%}')
print(f'  {"Hallucination Rate":<32} {hallu_rate:>9.1%}')
print(f'  {"-"*42}')
print(f'  {"Faithfulness (RAGAS)":<32} {avg_f:>10.4f}')
print(f'  {"Answer Relevancy (RAGAS)":<32} {avg_r:>10.4f}')
print(f'  {"Context Precision (RAGAS)":<32} {avg_cp:>10.4f}')
print(f'  {"Context Recall (RAGAS)":<32} {avg_cr:>10.4f}')
print('=' * 60)
print('\nPer-label accuracy:')
for lbl in ['yes','no','maybe']:
    sub = df[df['ground_truth']==lbl]
    if len(sub):
        print(f'  {lbl:>5}: {sub["is_correct"].mean():.1%} ({int(sub["is_correct"].sum())}/{len(sub)}) '
              f'| faithfulness={sub["faithfulness"].mean():.3f}')
print('\nBaris untuk tabel skripsi:')
print(f'  | Baseline (BM25) | {label_acc:.3f} | {hallu_rate:.3f} | '
      f'{avg_f:.3f} | {avg_r:.3f} | {avg_cp:.3f} | {avg_cr:.3f} |')
print('\n' + '-' * 60)
print(f'PENGINGAT: Jalankan ulang dengan MAX_SAMPLES=1000 untuk evaluasi final!')
print('-' * 60)